In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

In [2]:
reviews_df = pd.read_csv('data/movie_reviews.csv', index_col = 0)
reviews_df.head(1)

,review_id,movie_name,year,reviewer_name,review_text,rated,year_api,genre,directors,writers,actors,plot,first_genre,first_actor,first_director,first_writer,first_actor_gender,first_director_gender,first_writer_gender
0,1,Lethal Weapon 3,1992.0,J. Boyajian,"About 20 minutes into LETHAL WEAPON 3, my neph...",R,1992,"Action, Crime, Thriller",Richard Donner,"Jeffrey Boam, Robert Mark Kamen, Shane Black","Mel Gibson, Danny Glover, Joe Pesci",Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male


In [3]:
reviews_df['rated'].value_counts()

rated
R            8953
PG-13        4800
PG           1976
Not Rated     418
G             327
Approved      192
Passed        140
Unrated       118
NC-17         105
TV-MA          67
TV-PG          43
TV-14          26
X              14
GP             13
18+             3
TV-G            2
M/PG            2
13+             1
M               1
Name: count, dtype: int64

In [4]:
# -----------------------------
# 1. Rename inconsistent columns
# -----------------------------
def rename_columns(df):
    return df.rename({'year_api': 'movie_release_year'}, axis=1)


# -----------------------------
# 2. Split comma-separated metadata into lists
# -----------------------------
def split_to_list(series):
    return (
        series
        .fillna("")
        .astype(str)
        .apply(lambda x: [item.strip() for item in x.split(",") if item.strip()])
    )


# -----------------------------
# 3. Create structured metadata lists
# -----------------------------
def create_metadata_lists(df):
    df["genre_list"] = split_to_list(df["genre"])
    df["writer_list"] = split_to_list(df["writers"])
    df["director_list"] = split_to_list(df["directors"])
    df["actor_list"] = split_to_list(df["actors"])
    return df


# -----------------------------
# 4. Filter short / low-quality reviews
# -----------------------------
def filter_short_reviews(df, min_words=100):
    return df[
        df["review_text"].fillna("").str.split().str.len() >= min_words
    ]


# -----------------------------
# 5. Drop unused columns
# -----------------------------
def drop_unused_columns(df):
    return df.drop(
        ["review_id", "year", "genre", "writers", "actors", "directors"],
        axis=1,
        errors="ignore"
    )

# -----------------------------
# 6. Limit to allowed ratings
# -----------------------------
def filter_allowed_ratings(
    df,
    allowed_ratings=None
):
    if allowed_ratings is None:
        allowed_ratings = [
            "R",
            "PG-13",
            "PG",
            "Not Rated",
            "G"
        ]

    return df[df["rated"].isin(allowed_ratings)]

# -----------------------------
# 6. Full structural pipeline
# -----------------------------
def structural_cleaning_pipeline(df, min_words=100):
    """
    Non-text preprocessing stage for RAG:
    - schema normalization
    - metadata extraction
    - filtering
    - column cleanup
    """

    df = rename_columns(df)
    df = filter_allowed_ratings(df)
    df = create_metadata_lists(df)
    df = filter_short_reviews(df, min_words=min_words)
    df = drop_unused_columns(df)
    df["document_id"] = df.index.map(lambda i: f"movie_review_{i}")

    return df


In [5]:

# -----------------------------
# 1. Sentence-level removal of email/URL content
# -----------------------------
def remove_sentences_with_urls_or_emails(text):
    text = str(text)

    sentences = re.split(r'(?<=[.!?])\s+', text)

    cleaned = []
    for sent in sentences:
        if not re.search(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", sent) and \
           not re.search(r"(https?://|www\.)\S+", sent):
            cleaned.append(sent)

    return " ".join(cleaned).strip()


# -----------------------------
# 2. Direct removal of emails and URLs (safety net)
# -----------------------------
def remove_emails_and_urls(text):
    text = str(text)

    # emails
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "", text)

    # urls
    text = re.sub(r"(https?://|www\.)\S+", "", text)

    return text


# -----------------------------
# 3. Footer / boilerplate removal
# -----------------------------
def remove_footer_boilerplate(text):
    text = str(text)

    patterns = [
        r"Get the Internet just the way you want it.*",
        r"Free software.*",
        r"Free e-?mail.*",
        r"Try Juno Web.*",
        r"CompuServe.*",
        r"ICQ.*",
        r"AOL Instant Messenger.*",
        r"Sent via Deja\.com.*",
        r"Share what you know.*Learn what you don't.*",
    ]

    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    return text


# -----------------------------
# 4. Signature / author block removal
# -----------------------------
def remove_review_signatures(text):
    text = str(text)

    patterns = [
        r"\(C\)\s*\d{4}.*",
        r"For more reviews visit:.*",
        r"currently based in.*",
        r"has scripted and shot.*",
        r"ICQ\s*#\d+.*",
        r"on AOL Instant Messenger.*",
    ]

    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    return text


# -----------------------------
# 5. Core normalization (RAG-safe)
# -----------------------------
def normalize_text(text):
    text = str(text)

    # fix escaped characters
    text = text.replace("\\'", "'")
    text = text.replace('\\"', '"')

    # normalize quotes
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("’", "'")

    # remove control characters
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def remove_newsgroup_footer(text):
    text = str(text)

    patterns = [
        r"The review above was posted to.*",
        r"The Internet Movie Database accepts no responsibility.*",
        r"Unless stated otherwise, the copyright belongs to the author.*",
        r"Please direct comments/criticisms.*",
        r"Broken URLs in the reviews.*",
        r"Related links:.*",
        r"index of all rec\.arts\.movies\.reviews reviews.*",
        r"de\.rec\.film\.kritiken.*",
    ]

    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE | re.DOTALL)

    return text

In [6]:
def clean_text(text):
    text = str(text)

    # 1. remove structured footer blocks (IMPORTANT NEW STEP)
    text = remove_newsgroup_footer(text)

    # 2. remove sentences with URLs/emails (still useful earlier safeguard)
    text = remove_sentences_with_urls_or_emails(text)

    # 3. remove leftover URLs/emails
    text = remove_emails_and_urls(text)

    # 4. remove other boilerplate
    text = remove_footer_boilerplate(text)

    # 5. signature cleanup
    text = remove_review_signatures(text)

    # 6. normalize text
    text = normalize_text(text)

    return text

In [7]:
def full_cleaning_and_structure_pipeline(df):
    df = structural_cleaning_pipeline(df)
    df["review_text"] = df["review_text"].apply(clean_text)
    return df

In [8]:
reviews_df = full_cleaning_and_structure_pipeline(reviews_df)
reviews_df.head(5)

,movie_name,reviewer_name,review_text,rated,movie_release_year,plot,first_genre,first_actor,first_director,first_writer,first_actor_gender,first_director_gender,first_writer_gender,genre_list,writer_list,director_list,actor_list,document_id
0,Lethal Weapon 3,J. Boyajian,"About 20 minutes into LETHAL WEAPON 3, my neph...",R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_0
1,Lethal Weapon 3,Frank Maloney,LETHAL WEAPON 3 is a film directed by Richard ...,R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_1
2,Lethal Weapon 3,Brian L.,"120 min., R, Comedy/Action, 1992 Director: Ric...",R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_2
3,Lethal Weapon 3,Mark Santora,It has been a couple of years since we last sa...,R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_3
4,Lethal Weapon 3,Jose R.,I went to this movie with very low expectation...,R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Action,Mel Gibson,Richard Donner,Jeffrey Boam,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_4


In [16]:
# AdD Rag Text column
reviews_df["rag_text"] = (
    "Movie: " + reviews_df["movie_name"].fillna("") + "\n"
    "Rating: " + reviews_df["rated"].fillna("") + "\n"
    "Year: " + reviews_df["movie_release_year"].astype(str) + "\n"
    "Genres: " + reviews_df["genre_list"].apply(lambda x: ", ".join(x)) + "\n"
    "Actors: " + reviews_df["actor_list"].apply(lambda x: ", ".join(x)) + "\n"
    "Directors: " + reviews_df["director_list"].apply(lambda x: ", ".join(x)) + "\n"
    "Writers: " + reviews_df["writer_list"].apply(lambda x: ", ".join(x)) + "\n"
    "Plot: " + reviews_df["plot"].fillna("") + "\n"
    "Review: " + reviews_df["review_text"].fillna("")
)

In [18]:
print(reviews_df['rag_text'].iloc[0])

Movie: Lethal Weapon 3
Rating: R
Year: 1992
Genres: Action, Crime, Thriller
Actors: Mel Gibson, Danny Glover, Joe Pesci
Directors: Richard Donner
Writers: Jeffrey Boam, Robert Mark Kamen, Shane Black
Plot: Martin Riggs and Roger Murtaugh pursue a former LAPD officer who uses his knowledge of police procedure and policies to steal and sell confiscated guns and ammunition to local street gangs.
Review: About 20 minutes into LETHAL WEAPON 3, my nephew turned to me and asked, "Does this movie have a plot?" And that question represents everything that is wrong with LW3. Quite frankly, the movie is a mess on a number of levels. A *funny* mess, to be sure, but still a mess. As the Bard of Avon would put it, it was full of sound and fury, signifying nothing. It was just about everything a bad sequel usually is. o Joe Pesci's appearance in the film smacked of dollarsigns. Not just because he's "hot" at the moment, but because it seemed to exploit his contribution to LW2. His character and perfo

In [21]:
reviews_df_no_review_text = reviews_df.drop(['review_text', 'first_director', 'first_writer', 'first_genre'], axis = 1)
reviews_df_no_review_text.head(1)

,movie_name,reviewer_name,rated,movie_release_year,plot,first_actor,first_actor_gender,first_director_gender,first_writer_gender,genre_list,writer_list,director_list,actor_list,document_id,rag_text
0,Lethal Weapon 3,J. Boyajian,R,1992,Martin Riggs and Roger Murtaugh pursue a forme...,Mel Gibson,female,male,male,"[Action, Crime, Thriller]","[Jeffrey Boam, Robert Mark Kamen, Shane Black]",[Richard Donner],"[Mel Gibson, Danny Glover, Joe Pesci]",movie_review_0,Movie: Lethal Weapon 3\nRating: R\nYear: 1992\...


In [22]:
reviews_df_no_review_text.to_csv('data/cleaned_movie_reviews.csv')

list